# 数据处理标准

- 高缺失率（80%以上）列不插值，在XGBoost、RF建模时直接带NaN值建模，对别的模型，训练前加一句删除带缺失值的特征；

**【结果】**

- df：带NaN，不插值；

- df_NaN_fixed：不带NaN，经过去缺或插值。

## 读取数据

In [1]:
import numpy as np

In [2]:
import pandas as pd

# 读取 CSV 文件，默认编码为 utf-8，如果出错可尝试 encoding='gbk'
df = pd.read_csv('../data/data.csv', encoding='utf-8')

In [3]:
# 查看列数
print(f"列数: {df.shape[1]}")

列数: 59


In [4]:
df.head()

,Group,Hos_name,Age,Height,Weight,BMI,Marriage,GPA,Guilv_Menstruation,Guilv_Menstruation_max,...,E2,P,T,PLT_Ratio,NLR,PLR,SII,Gravida,Para,Abortion
0,1,GYYY,20,1.65,50.0,18.37,single,G0P0A0,28-30,30.0,...,NaN,NaN,NaN,NaN,1.11,105.00,210.00,0.0,0.0,0.0
1,1,GYYY,38,1.60,50.0,19.53,single,G1P0A1,28,28.0,...,NaN,NaN,NaN,NaN,2.68,175.26,893.84,1.0,0.0,1.0
2,1,GYYY,36,1.55,60.0,24.97,Married,G1P1A0,30-32,32.0,...,NaN,NaN,NaN,NaN,3.00,160.00,624.00,1.0,1.0,0.0
3,1,GYYY,36,1.60,70.0,27.34,Married,G1P1A0,28-30,30.0,...,NaN,NaN,NaN,NaN,1.54,124.23,496.92,1.0,1.0,0.0
4,1,GYYY,25,1.53,60.0,25.63,single,G0P0A0,28-30,30.0,...,63.0,5.64,0.43,NaN,2.48,178.57,928.57,0.0,0.0,0.0


### 删除100%缺失列

如果一个列在所有Group值为Control的行中都为NaN，则从df中删除该列

In [5]:
# 找到所有Group为Control的行
control_rows = df[df['Group'] == 0]

# 找出这些行中全为NaN的列
cols_all_nan_in_control = control_rows.columns[control_rows.isna().all()]

# 从df中删除这些列
df = df.drop(columns=cols_all_nan_in_control)

In [8]:
print(cols_all_nan_in_control)

Index(['Height', 'Weight', 'BMI', 'Marriage', 'GPA', 'Guilv_Menstruation',
       'Guilv_Menstruation_max', 'Length_Menstruation_max',
       'Length_Menstruation', 'Tongjing_YN', 'D2', 'PCT', 'AMH', 'PLT_Ratio',
       'Gravida', 'Para', 'Abortion'],
      dtype='object')


In [6]:
cols = df.columns.tolist()
for i in range(0, len(cols), 5):
    print(cols[i:i+5])

['Group', 'Hos_name', 'Age', 'WBC', 'NEUT_No']
['LYM_No', 'MON_No', 'EOS_No', 'BASO_No', 'NRBC_No']
['RBC', 'HGB', 'HCT', 'MCV', 'MCH']
['MCHC', 'RDW-SD', 'RDW-CV', 'PLT', 'MPV']
['PDW', 'Plateletcrit', 'PT', 'INR', 'PTA']
['FIB', 'APTT', 'PTT_ratio', 'TT', 'CRP']
['CA125', 'CA199', 'HE4', 'FSH', 'LH']
['PRL', 'E2', 'P', 'T', 'NLR']
['PLR', 'SII']


In [7]:
summary = pd.DataFrame({
    '列名': df.columns,
    '数据类型': df.dtypes.values,
    '缺失值数量': df.isnull().sum().values,
    '缺失率(%)': (df.isnull().mean() * 100).round(2).values
})

print(summary)

              列名     数据类型  缺失值数量  缺失率(%)
0          Group    int64      0    0.00
1       Hos_name   object      0    0.00
2            Age    int64      0    0.00
3            WBC  float64    146    3.17
4        NEUT_No  float64    146    3.17
5         LYM_No  float64    146    3.17
6         MON_No  float64    148    3.21
7         EOS_No  float64    148    3.21
8        BASO_No  float64    148    3.21
9        NRBC_No  float64   1466   31.83
10           RBC  float64    148    3.21
11           HGB  float64    146    3.17
12           HCT  float64    149    3.24
13           MCV  float64    148    3.21
14           MCH  float64    148    3.21
15          MCHC  float64    149    3.24
16        RDW-SD  float64   1774   38.52
17        RDW-CV  float64    148    3.21
18           PLT  float64    146    3.17
19           MPV  float64    157    3.41
20           PDW  float64    157    3.41
21  Plateletcrit  float64    180    3.91
22            PT  float64   2311   50.18
23           INR

### ['Group', 'No', 'Center', 'Age', 'WBC']

In [11]:
import pandas as pd

selected_cols = ['Group', 'Hos_name', 'Age', 'WBC']  # ← 替换为你的列名

for col in selected_cols:
    print(f"\n列名: {col}")
    print(f"数据类型: {df[col].dtype}")
    
    missing_count = df[col].isnull().sum()
    missing_pct = df[col].isnull().mean() * 100
    print(f"缺失值数量: {missing_count}  缺失率: {missing_pct:.2f}%")
    
    # 判断是否为分类变量（新版写法）
    if isinstance(df[col].dtype, pd.CategoricalDtype) or df[col].dtype == 'object':
        print("取值频数:")
        print(df[col].value_counts(dropna=False))
    elif pd.api.types.is_numeric_dtype(df[col]):
        print("描述性统计:")
        print(df[col].describe())
    else:
        print("⚠️ 非常见类型，无法分析")


列名: Group
数据类型: int64
缺失值数量: 0  缺失率: 0.00%
描述性统计:
count    4605.000000
mean        0.467970
std         0.499027
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000
Name: Group, dtype: float64

列名: Hos_name
数据类型: object
缺失值数量: 0  缺失率: 0.00%
取值频数:
GYYY         3277
QYSRMYY      1155
SZLHQRMYY     173
Name: Hos_name, dtype: int64

列名: Age
数据类型: int64
缺失值数量: 0  缺失率: 0.00%
描述性统计:
count    4605.000000
mean       35.013246
std         7.632812
min        16.000000
25%        29.000000
50%        35.000000
75%        41.000000
max        55.000000
Name: Age, dtype: float64

列名: WBC
数据类型: float64
缺失值数量: 146  缺失率: 3.17%
描述性统计:
count    4459.000000
mean        6.402597
std         1.677705
min         2.240000
25%         5.300000
50%         6.200000
75%         7.280000
max        40.550000
Name: WBC, dtype: float64


**留待中位填充处理**

### 【【弃用】】GPA列处理

由于在Control中全缺，因此GPA列已经删除，该部分弃用

解析GPA列

对解析出的4列进行描述性统计

缺失列情况：所有Control都没有这个数值

### ['NEUT_No', 'LYM_No', 'MON_No', 'EOS_No', 'BASO_No']

In [9]:
import pandas as pd

selected_cols = ['NEUT_No', 'LYM_No', 'MON_No', 'EOS_No', 'BASO_No']  # ← 替换为你的列名

for col in selected_cols:
    print(f"\n列名: {col}")
    print(f"数据类型: {df[col].dtype}")
    
    missing_count = df[col].isnull().sum()
    missing_pct = df[col].isnull().mean() * 100
    print(f"缺失值数量: {missing_count}  缺失率: {missing_pct:.2f}%")
    
    # 判断是否为分类变量（新版写法）
    if isinstance(df[col].dtype, pd.CategoricalDtype) or df[col].dtype == 'object':
        print("取值频数:")
        print(df[col].value_counts(dropna=False))
    elif pd.api.types.is_numeric_dtype(df[col]):
        print("描述性统计:")
        print(df[col].describe())
    else:
        print("⚠️ 非常见类型，无法分析")


列名: NEUT_No
数据类型: float64
缺失值数量: 146  缺失率: 3.17%
描述性统计:
count    4459.000000
mean        3.800899
std         1.348640
min         0.870000
25%         2.900000
50%         3.600000
75%         4.500000
max        20.210000
Name: NEUT_No, dtype: float64

列名: LYM_No
数据类型: float64
缺失值数量: 146  缺失率: 3.17%
描述性统计:
count    4459.000000
mean        1.983842
std         0.516401
min         0.000000
25%         1.600000
50%         1.900000
75%         2.300000
max         4.100000
Name: LYM_No, dtype: float64

列名: MON_No
数据类型: float64
缺失值数量: 148  缺失率: 3.21%
描述性统计:
count    4457.000000
mean        0.427401
std         0.144287
min         0.000000
25%         0.300000
50%         0.400000
75%         0.500000
max         3.900000
Name: MON_No, dtype: float64

列名: EOS_No
数据类型: float64
缺失值数量: 148  缺失率: 3.21%
描述性统计:
count    4457.000000
mean        0.145363
std         0.127558
min         0.000000
25%         0.080000
50%         0.110000
75%         0.200000
max         2.720000
Name: EOS_No, d

#### 评价

缺失率都很低，数据类型没问题，留待中位填充

### ['NRBC_No', 'RBC', 'HGB', 'HCT', 'MCV']

In [12]:
import pandas as pd

selected_cols = ['NRBC_No', 'RBC', 'HGB', 'HCT', 'MCV']  # ← 替换为你的列名

for col in selected_cols:
    print(f"\n列名: {col}")
    print(f"数据类型: {df[col].dtype}")
    
    missing_count = df[col].isnull().sum()
    missing_pct = df[col].isnull().mean() * 100
    print(f"缺失值数量: {missing_count}  缺失率: {missing_pct:.2f}%")
    
    # 判断是否为分类变量（新版写法）
    if isinstance(df[col].dtype, pd.CategoricalDtype) or df[col].dtype == 'object':
        print("取值频数:")
        print(df[col].value_counts(dropna=False))
    elif pd.api.types.is_numeric_dtype(df[col]):
        print("描述性统计:")
        print(df[col].describe())
    else:
        print("⚠️ 非常见类型，无法分析")


列名: NRBC_No
数据类型: float64
缺失值数量: 1466  缺失率: 31.83%
描述性统计:
count    3139.000000
mean        0.004876
std         0.010131
min         0.000000
25%         0.000000
50%         0.003000
75%         0.009000
max         0.293000
Name: NRBC_No, dtype: float64

列名: RBC
数据类型: float64
缺失值数量: 148  缺失率: 3.21%
描述性统计:
count    4457.000000
mean        4.476416
std         0.461421
min         1.410000
25%         4.190000
50%         4.420000
75%         4.690000
max         6.670000
Name: RBC, dtype: float64

列名: HGB
数据类型: float64
缺失值数量: 146  缺失率: 3.17%
描述性统计:
count    4459.000000
mean      124.115945
std        14.341670
min        55.000000
25%       118.000000
50%       127.000000
75%       133.000000
max       165.000000
Name: HGB, dtype: float64

列名: HCT
数据类型: float64
缺失值数量: 149  缺失率: 3.24%
描述性统计:
count    4456.000000
mean        0.378438
std         0.036231
min         0.200000
25%         0.361000
50%         0.383000
75%         0.401000
max         0.750000
Name: HCT, dtype: float64

列

#### NRBC_No

In [13]:
# 缺失个数与总数
group_stats = df.groupby('Group')['NRBC_No'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
1               1343      0.623202
0                123      0.050204


#### HCT

#### RBC

正常4-6，大于100的和小于1的置为NaN

#### HGB

血红蛋白，正常值范围都是在一两百，这里采取直接去掉＜1和＞1000的数值

#### MCV

80-100都正常，数据没啥问题，留待补缺失

#### 评价

- NRBC_No列Endo组缺失超过50%，对照列缺失在5%以下，总体属分组偏倚缺失，难以处理，建议XGBoost原样训练，其余模型删除该列；

- 均留待补充缺失值

### ['MCH', 'MCHC', 'RDW_SD', 'RDW_CV', 'PLT']

In [15]:
# Checkpoint1 保存df
df.to_pickle('../checkpoint/df_checkpoint1.pkl')  # 保存

In [17]:
selected_cols = ['MCH', 'MCHC', 'RDW-SD', 'RDW-CV', 'PLT']  # ← 替换为你的列名

for col in selected_cols:
    print(f"\n列名: {col}")
    print(f"数据类型: {df[col].dtype}")
    
    missing_count = df[col].isnull().sum()
    missing_pct = df[col].isnull().mean() * 100
    print(f"缺失值数量: {missing_count}  缺失率: {missing_pct:.2f}%")
    
    # 判断是否为分类变量（新版写法）
    if isinstance(df[col].dtype, pd.CategoricalDtype) or df[col].dtype == 'object':
        print("取值频数:")
        print(df[col].value_counts(dropna=False))
    elif pd.api.types.is_numeric_dtype(df[col]):
        print("描述性统计:")
        print(df[col].describe())
    else:
        print("⚠️ 非常见类型，无法分析")


列名: MCH
数据类型: float64
缺失值数量: 148  缺失率: 3.21%
描述性统计:
count    4457.000000
mean       27.913956
std         3.593318
min        12.000000
25%        26.900000
50%        29.100000
75%        30.300000
max        36.400000
Name: MCH, dtype: float64

列名: MCHC
数据类型: float64
缺失值数量: 149  缺失率: 3.24%
描述性统计:
count    4456.000000
mean      327.395197
std        13.563345
min       247.000000
25%       321.000000
50%       330.000000
75%       336.000000
max       368.000000
Name: MCHC, dtype: float64

列名: RDW-SD
数据类型: float64
缺失值数量: 1774  缺失率: 38.52%
描述性统计:
count    2831.000000
mean       41.601973
std         3.429882
min        31.500000
25%        39.810000
50%        41.560000
75%        42.900000
max        76.100000
Name: RDW-SD, dtype: float64

列名: RDW-CV
数据类型: float64
缺失值数量: 148  缺失率: 3.21%
描述性统计:
count    4457.000000
mean       13.830245
std         1.990058
min        10.600000
25%        12.700000
50%        13.200000
75%        14.200000
max        31.100000
Name: RDW-CV, dtype: floa

#### MCH

正常值27-31，小于1和大于200的置空

#### MCHC

320-360为正常值，看着都还不错，pass

#### RDW_SD

In [18]:
# 缺失个数与总数
group_stats = df.groupby('Group')['RDW-SD'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
1               1653      0.767053
0                121      0.049388


#### RDW_CV

未见异常

#### PLT

血小板，100-300为正常值，孕妇可能会有高凝；去掉10以下的

#### 评价

- RDW_SD：Endo组缺失超过50%，对照列缺失在5%以下，总体属分组偏倚缺失，难以处理，原模型未使用该值；本轮可考虑用于XGboost

- RDW_CV没见异常值，上轮原模型未使用，本轮考虑正常使用

### ['MPV', 'PDW', 'Plateletcrit', 'PT', 'INR']

In [19]:
# ['MPV', 'PDW', 'PLT_Yaji', 'PT', 'INR']

selected_cols = ['MPV', 'PDW', 'Plateletcrit', 'PT', 'INR']  # ← 替换为你的列名

for col in selected_cols:
    print(f"\n列名: {col}")
    print(f"数据类型: {df[col].dtype}")
    
    missing_count = df[col].isnull().sum()
    missing_pct = df[col].isnull().mean() * 100
    print(f"缺失值数量: {missing_count}  缺失率: {missing_pct:.2f}%")
    
    # 判断是否为分类变量（新版写法）
    if isinstance(df[col].dtype, pd.CategoricalDtype) or df[col].dtype == 'object':
        print("取值频数:")
        print(df[col].value_counts(dropna=False))
    elif pd.api.types.is_numeric_dtype(df[col]):
        print("描述性统计:")
        print(df[col].describe())
    else:
        print("⚠️ 非常见类型，无法分析")


列名: MPV
数据类型: float64
缺失值数量: 157  缺失率: 3.41%
描述性统计:
count    4448.000000
mean        8.822482
std         1.103481
min         5.900000
25%         8.000000
50%         8.700000
75%         9.500000
max        16.000000
Name: MPV, dtype: float64

列名: PDW
数据类型: float64
缺失值数量: 157  缺失率: 3.41%
描述性统计:
count    4448.000000
mean       15.872050
std         1.767028
min         7.700000
25%        15.900000
50%        16.200000
75%        16.800000
max        24.600000
Name: PDW, dtype: float64

列名: Plateletcrit
数据类型: float64
缺失值数量: 180  缺失率: 3.91%
描述性统计:
count    4425.000000
mean        0.234788
std         0.057697
min         0.060000
25%         0.196000
50%         0.226000
75%         0.265000
max         0.640000
Name: Plateletcrit, dtype: float64

列名: PT
数据类型: float64
缺失值数量: 2311  缺失率: 50.18%
描述性统计:
count    2294.000000
mean       12.778248
std         1.137998
min         9.300000
25%        12.200000
50%        12.900000
75%        13.500000
max        26.400000
Name: PT, dtype: fl

#### MPV

血小板平均体积，正常值7-11，大于等于40的置空

#### PDW

血小板分布宽度，10-15左右属于正常，小于1的置空

#### PLT_Yaji

血小板压积，0.12-0.32左右，删除大于10的

#### PT

凝血酶原时间PT，10-14左右正常，大于100的置空

In [20]:
# 缺失个数与总数
group_stats = df.groupby('Group')['PT'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               2291      0.935102
1                 20      0.009281


#### INR

PT国际标准化比值，0.8-1.2左右，等于0的和大于20的置空



In [21]:
# 缺失个数与总数
group_stats = df.groupby('Group')['INR'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               2292      0.935510
1                198      0.091879


#### 评价

- PT和INR：Endo组缺失超过90%，对照列缺失在5%以下，总体属分组偏倚缺失，难以处理

### ['PTA', 'FIB', 'APTT', 'PTT_ratio', 'TT']

In [23]:
# Checkpoint2 保存df
df.to_pickle('../checkpoint/df_checkpoint2.pkl')  # 保存

In [24]:
# ['PTA', 'FIB', 'APTT', 'PTT_ratio', 'TT']
selected_cols = ['PTA', 'FIB', 'APTT', 'PTT_ratio', 'TT']  # ← 替换为你的列名

for col in selected_cols:
    print(f"\n列名: {col}")
    print(f"数据类型: {df[col].dtype}")
    
    missing_count = df[col].isnull().sum()
    missing_pct = df[col].isnull().mean() * 100
    print(f"缺失值数量: {missing_count}  缺失率: {missing_pct:.2f}%")
    
    # 判断是否为分类变量（新版写法）
    if isinstance(df[col].dtype, pd.CategoricalDtype) or df[col].dtype == 'object':
        print("取值频数:")
        print(df[col].value_counts(dropna=False))
    elif pd.api.types.is_numeric_dtype(df[col]):
        print("描述性统计:")
        print(df[col].describe())
    else:
        print("⚠️ 非常见类型，无法分析")



列名: PTA
数据类型: float64
缺失值数量: 2488  缺失率: 54.03%
描述性统计:
count    2117.000000
mean       96.490203
std        15.617022
min        50.600000
25%        86.200000
50%        95.000000
75%       105.000000
max       212.100000
Name: PTA, dtype: float64

列名: FIB
数据类型: float64
缺失值数量: 2311  缺失率: 50.18%
描述性统计:
count    2294.000000
mean        2.872847
std         0.668236
min         0.840000
25%         2.420000
50%         2.800000
75%         3.220000
max         7.070000
Name: FIB, dtype: float64

列名: APTT
数据类型: float64
缺失值数量: 2311  缺失率: 50.18%
描述性统计:
count    2294.000000
mean       33.325131
std         5.703093
min        17.900000
25%        28.800000
50%        32.700000
75%        37.500000
max        62.300000
Name: APTT, dtype: float64

列名: PTT_ratio
数据类型: float64
缺失值数量: 2494  缺失率: 54.16%
描述性统计:
count    2111.000000
mean        1.085088
std         0.116475
min         0.740000
25%         1.020000
50%         1.070000
75%         1.140000
max         2.650000
Name: PTT_ratio, dtype

#### PTA

凝血酶原活动度，75-100左右为正常，看下来无极端异常值

缺失严重，先看下分布

In [25]:
# 缺失个数与总数
group_stats = df.groupby('Group')['PTA'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               2292      0.935510
1                196      0.090951


严重偏倚的缺失，非XGBoost的话不使用这列。

#### FIB

纤维蛋白原，正常值2-4，超过15的置空

In [26]:
# 缺失个数与总数
group_stats = df.groupby('Group')['FIB'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               2291      0.935102
1                 20      0.009281


同上。

#### APTT

活化部分凝血活酶时间，正常值为25-35左右，小于1的置空

In [27]:
# 缺失个数与总数
group_stats = df.groupby('Group')['APTT'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               2291      0.935102
1                 20      0.009281


同上。

#### PTT_ratio

PTT比率，正常值不详，考虑大于10置空

In [28]:
# 缺失个数与总数
group_stats = df.groupby('Group')['PTT_ratio'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               2292      0.935510
1                202      0.093735


#### TT

凝血时间TT，正常值为14-21，小于1置空

In [29]:
# 缺失个数与总数
group_stats = df.groupby('Group')['TT'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               2291      0.935102
1                 20      0.009281


### ['CRP', 'CA125', 'CA199', 'HE4', 'FSH']

In [30]:
# Checkpoint3 保存df
df.to_pickle('../checkpoint/df_checkpoint3.pkl')  # 保存

In [31]:
# ['CRP', 'CA125', 'CA199', 'HE4', 'FSH']

selected_cols = ['CRP', 'CA125', 'CA199', 'HE4', 'FSH']  # ← 替换为你的列名

for col in selected_cols:
    print(f"\n列名: {col}")
    print(f"数据类型: {df[col].dtype}")
    
    missing_count = df[col].isnull().sum()
    missing_pct = df[col].isnull().mean() * 100
    print(f"缺失值数量: {missing_count}  缺失率: {missing_pct:.2f}%")
    
    # 判断是否为分类变量（新版写法）
    if isinstance(df[col].dtype, pd.CategoricalDtype) or df[col].dtype == 'object':
        print("取值频数:")
        print(df[col].value_counts(dropna=False))
    elif pd.api.types.is_numeric_dtype(df[col]):
        print("描述性统计:")
        print(df[col].describe())
    else:
        print("⚠️ 非常见类型，无法分析")


列名: CRP
数据类型: float64
缺失值数量: 4401  缺失率: 95.57%
描述性统计:
count    204.000000
mean       4.451127
std       20.462675
min        0.000000
25%        0.050000
50%        0.105000
75%        0.515000
max      173.200000
Name: CRP, dtype: float64

列名: CA125
数据类型: float64
缺失值数量: 1963  缺失率: 42.63%
描述性统计:
count    2642.000000
mean       46.923978
std       135.913072
min         1.860000
25%        11.700000
50%        20.715000
75%        48.277500
max      5414.000000
Name: CA125, dtype: float64

列名: CA199
数据类型: float64
缺失值数量: 3474  缺失率: 75.44%
描述性统计:
count    1131.000000
mean       25.664164
std        45.925186
min         0.600000
25%         9.100000
50%        15.540000
75%        27.605000
max      1000.000000
Name: CA199, dtype: float64

列名: HE4
数据类型: float64
缺失值数量: 3246  缺失率: 70.49%
描述性统计:
count    1359.000000
mean       53.427564
std        14.223537
min         3.380000
25%        44.735000
50%        51.260000
75%        59.450000
max       251.300000
Name: HE4, dtype: float64

列名:

#### CRP

缺失很严重，要不了了

In [32]:
# 缺失个数与总数
group_stats = df.groupby('Group')['CRP'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
1               2092      0.970766
0               2309      0.942449


在非XGBoost中去掉这列

异常值不详，未去

#### CA125

糖蛋白性肿瘤相关抗原125，正常值0-35，异常值可极高，因此不做处理

In [33]:
# 缺失个数与总数
group_stats = df.groupby('Group')['CA125'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               1602      0.653878
1                361      0.167517


偏Control缺失，考虑中位数补缺

#### CA199

糖蛋白性肿瘤相关抗原199，正常值0-37，异常值可极高，因此不做处理

In [34]:
# 缺失个数与总数
group_stats = df.groupby('Group')['CA199'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               2301      0.939184
1               1173      0.544316


偏Control缺失，考虑中位数补缺

#### HE4

人附睾蛋白4，0-150为正常范围

In [35]:
# 缺失个数与总数
group_stats = df.groupby('Group')['HE4'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               2412      0.984490
1                834      0.387007


无异常值

#### FSH

促卵泡激素，0-150均为可能出现的值，异常值可高

In [36]:
# 缺失个数与总数
group_stats = df.groupby('Group')['FSH'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               2433      0.993061
1               1272      0.590255


### ['LH', 'PRL', 'E2', 'P', 'T']

In [37]:
# ['LH', 'PRL', 'E2', 'P', 'T']

selected_cols = ['LH', 'PRL', 'E2', 'P', 'T']  # ← 替换为你的列名

for col in selected_cols:
    print(f"\n列名: {col}")
    print(f"数据类型: {df[col].dtype}")
    
    missing_count = df[col].isnull().sum()
    missing_pct = df[col].isnull().mean() * 100
    print(f"缺失值数量: {missing_count}  缺失率: {missing_pct:.2f}%")
    
    # 判断是否为分类变量（新版写法）
    if isinstance(df[col].dtype, pd.CategoricalDtype) or df[col].dtype == 'object':
        print("取值频数:")
        print(df[col].value_counts(dropna=False))
    elif pd.api.types.is_numeric_dtype(df[col]):
        print("描述性统计:")
        print(df[col].describe())
    else:
        print("⚠️ 非常见类型，无法分析")


列名: LH
数据类型: float64
缺失值数量: 3705  缺失率: 80.46%
描述性统计:
count    900.000000
mean      11.766652
std       12.244201
min        0.000000
25%        5.005000
50%        8.000000
75%       12.770000
max       96.800000
Name: LH, dtype: float64

列名: PRL
数据类型: float64
缺失值数量: 3741  缺失率: 81.24%
描述性统计:
count    864.000000
mean      57.927080
std      115.573614
min        0.707000
25%       15.460000
50%       25.460000
75%       38.582500
max      933.100000
Name: PRL, dtype: float64

列名: E2
数据类型: float64
缺失值数量: 3667  缺失率: 79.63%
描述性统计:
count    938.000000
mean     140.149851
std      123.857265
min        0.010000
25%       56.550000
50%      103.500000
75%      186.750000
max      895.000000
Name: E2, dtype: float64

列名: P
数据类型: float64
缺失值数量: 3721  缺失率: 80.80%
描述性统计:
count    884.000000
mean       2.467420
std        4.458599
min        0.030000
25%        0.122500
50%        0.316500
75%        2.152500
max       29.300000
Name: P, dtype: float64

列名: T
数据类型: float64
缺失值数量: 3742  缺失率: 81.26

#### LH

黄体生成激素（LH），正常值为0-100多差不多。

In [38]:
# 缺失个数与总数
group_stats = df.groupby('Group')['LH'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               2433      0.993061
1               1272      0.590255


无异常值

#### PRL

催乳素（PRL），0-400均有可能，可高，不去除异常值

In [39]:
# 缺失个数与总数
group_stats = df.groupby('Group')['PRL'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               2433      0.993061
1               1308      0.606961


不去异常值

#### E2

雌二醇（E2），高值可2000以上，不去异常值

In [40]:
# 缺失个数与总数
group_stats = df.groupby('Group')['E2'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               2391      0.975918
1               1276      0.592111


#### P

孕酮（P），0-1000多都可能出现

In [41]:
# 缺失个数与总数
group_stats = df.groupby('Group')['P'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               2433      0.993061
1               1288      0.597680


无异常值

#### T

睾酮（T），无异常值

In [42]:
# 缺失个数与总数
group_stats = df.groupby('Group')['T'].agg(['count', 'size'])
group_stats['missing_count'] = group_stats['size'] - group_stats['count']
group_stats['missing_rate'] = group_stats['missing_count'] / group_stats['size']

print(group_stats[['missing_count', 'missing_rate']].sort_values('missing_rate', ascending=False))

       missing_count  missing_rate
Group                             
0               2433      0.993061
1               1309      0.607425
